# 02 – Preprocess SD7K (Kaggle / local CPU)
Resize SD7K pairs to 512×512 JPEG (quality 95) and split by document ID.

**Run this on Kaggle**, then save output as a Kaggle Dataset `sd7k-512`.

Expected raw folder layout:
```
sd7k_raw/
  train/input/      ← shadow images (.png or .jpg)
  train/target/     ← shadow-free images (.png or .jpg)
  test/input/
  test/target/
```

In [ ]:
import os, sys, shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import random
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

# Supported input extensions (dataset uses PNG or JPG)
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.PNG', '.JPG', '.JPEG'}

INPUT_ALIASES = {'input', 'inputs', 'shadow', 'shadows', 'shadow_img', 'shadow_images', 'input_img', 'input_images'}
TARGET_ALIASES = {'target', 'targets', 'gt', 'clean', 'shadow_free', 'shadow-free', 'shadowfree', 'ground_truth', 'groundtruth', 'target_img', 'target_images'}

def glob_images(directory: Path):
    """Return all image files in a directory, safely handling non-existent paths."""
    if not directory or not directory.exists() or not directory.is_dir():
        return []
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in {e.lower() for e in IMG_EXTS})

def get_split_dir(root: Path, split: str, subdir: str) -> Path:
    """Resolve split and subdir flexibly (handling aliases and case-insensitivity)."""
    if not root.exists():
        return root / split / subdir

    # 1. Resolve split directory (train, val, test)
    split_dir = None
    for s in root.iterdir():
        if s.is_dir() and s.name.lower() == split.lower():
            split_dir = s
            break
    if split_dir is None:
        return root / split / subdir

    # 2. Resolve subdir (input vs target)
    is_input = subdir.lower() in INPUT_ALIASES
    target_aliases = INPUT_ALIASES if is_input else TARGET_ALIASES
    
    # Check exact
    exact = split_dir / subdir
    if exact.exists():
        return exact

    # Check aliases
    for sub in split_dir.iterdir():
        if sub.is_dir() and sub.name.lower() in target_aliases:
            return sub

    return split_dir / subdir

def build_target_lookup(target_dir: Path) -> dict:
    """Build lookup dict for fast, flexible matching across extensions and casing."""
    lookup = {}
    if not target_dir or not target_dir.exists():
        return lookup
    for p in target_dir.iterdir():
        if p.is_file() and p.suffix.lower() in {e.lower() for e in IMG_EXTS}:
            lookup[p.name] = p
            lookup[p.stem] = p
            lookup[p.stem.lower()] = p
    return lookup

def find_matching_target(inp_path: Path, target_lookup: dict) -> Path:
    """Find the corresponding target image path for an input image."""
    # Exact name match
    if inp_path.name in target_lookup:
        return target_lookup[inp_path.name]
    # Exact stem match (e.g. input is .png, target is .jpg)
    if inp_path.stem in target_lookup:
        return target_lookup[inp_path.stem]
    # Case-insensitive stem match
    if inp_path.stem.lower() in target_lookup:
        return target_lookup[inp_path.stem.lower()]
    # Strip _shadow / _input suffix
    clean_stem = inp_path.stem.lower().replace('_shadow', '').replace('_input', '')
    if clean_stem in target_lookup:
        return target_lookup[clean_stem]
    return None

def find_sd7k_root() -> Path:
    """Auto-detect SD7K dataset root directory on Kaggle, local, or Colab."""
    search_roots = [Path('/kaggle/input'), Path('data'), Path('.')]
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base]
        try:
            for p in base.rglob('*'):
                if p.is_dir():
                    candidates.append(p)
        except Exception:
            pass
        for c in candidates:
            try:
                for child in c.iterdir():
                    if child.is_dir() and child.name.lower() in ('train', 'training'):
                        for subchild in child.iterdir():
                            if subchild.is_dir() and subchild.name.lower() in INPUT_ALIASES:
                                for img in subchild.iterdir():
                                    if img.suffix.lower() in {e.lower() for e in IMG_EXTS}:
                                        return c
            except Exception:
                continue
    return None

# ── Paths ──────────────────────────────────────────────────────────────────
KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    detected = find_sd7k_root()
    if detected:
        SD7K_RAW = detected
        print(f'✅ Auto-detected SD7K dataset at: {SD7K_RAW}')
    else:
        SD7K_RAW = Path('/kaggle/input/sd7k-raw')
        print(f'⚠️ SD7K not auto-detected. Defaulting to: {SD7K_RAW}')
        if Path('/kaggle/input').exists():
            print('Available paths in /kaggle/input:')
            for p in sorted(Path('/kaggle/input').rglob('*'))[:30]:
                print(f'   {p}')
    OUT_DIR = Path('/kaggle/working/sd7k_512')
else:
    SD7K_RAW = Path('data/sd7k_raw')
    OUT_DIR  = Path('data/sd7k_512')

IMAGE_SIZE = 512
JPEG_Q     = 95
VAL_FRAC   = 0.05
SEED       = 42

print(f'SD7K raw: {SD7K_RAW}')
print(f'Output:   {OUT_DIR}')

In [ ]:
# ── Inspect folder structure ───────────────────────────────────────────────
found_any = False
for split in ['train', 'test']:
    for subdir in ['input', 'target']:
        p = get_split_dir(SD7K_RAW, split, subdir)
        n = len(glob_images(p))
        if n > 0:
            found_any = True
        print(f'  {split}/{subdir}: {n} images (checked: {p})')

train_in = get_split_dir(SD7K_RAW, 'train', 'input')
sample_in_files = glob_images(train_in)[:5]
print('\nSample input filenames:', [f.name for f in sample_in_files])

train_tgt = get_split_dir(SD7K_RAW, 'train', 'target')
sample_tgt_files = glob_images(train_tgt)[:5]
print('Sample target filenames:', [f.name for f in sample_tgt_files])

if not found_any:
    print('\n❌ WARNING: No images found at expected paths.')
    print('Listing contents of /kaggle/input:')
    if Path('/kaggle/input').exists():
        for p in sorted(Path('/kaggle/input').rglob('*'))[:30]:
            print(f'   {p}')

In [ ]:
# ── Extract document IDs and pair matching ────────────────────────────────
# IMPORTANT: Split by document ID, not by image, to avoid data leakage.
import re

def get_doc_id(filename: str) -> str:
    """Extract document ID from filename.
    
    Common SD7K formats:
      '0001_001.png'     → doc_id = '0001'
      'doc001_cap01.png' → doc_id = 'doc001'
      'IMG_0002.png'     → doc_id = '0002'
    """
    m = re.match(r'^(?:IMG_)?(.+?)(?:_.*)?$', Path(filename).stem, re.IGNORECASE)
    return m.group(1) if m else filename

train_in = get_split_dir(SD7K_RAW, 'train', 'input')
train_tgt = get_split_dir(SD7K_RAW, 'train', 'target')

tgt_lookup = build_target_lookup(train_tgt)
all_in_files = glob_images(train_in)

# Pair input files with their matching target file
paired_files = []
unpaired = []
for inp in all_in_files:
    tgt = find_matching_target(inp, tgt_lookup)
    if tgt is not None:
        paired_files.append((inp, tgt))
    else:
        unpaired.append(inp.name)

if not paired_files:
    raise FileNotFoundError(
        f"No paired images found between:\n"
        f"  Input:  {train_in} ({len(all_in_files)} images)\n"
        f"  Target: {train_tgt} ({len(tgt_lookup)} images)\n"
        f"Sample inputs:  {[f.name for f in all_in_files[:5]]}\n"
        f"Sample targets: {list(tgt_lookup.keys())[:5]}"
    )

print(f'✅ Successfully matched {len(paired_files)} training pairs.')
if unpaired:
    print(f'⚠️ Notice: {len(unpaired)} input images had no matching target and will be safely skipped (e.g. {unpaired[:3]})')

doc_ids = sorted(set(get_doc_id(inp.name) for inp, _ in paired_files))
print(f'Found {len(doc_ids)} unique document IDs across {len(paired_files)} training pairs')
print(f'First 10 doc IDs: {doc_ids[:10]}')

In [ ]:
# ── Train/Val split by document ID ────────────────────────────────────────
random.seed(SEED)
n_val_docs = max(1, int(len(doc_ids) * VAL_FRAC))
val_docs = set(random.sample(doc_ids, n_val_docs))
train_docs = set(doc_ids) - val_docs

print(f'Train docs: {len(train_docs)}, Val docs: {len(val_docs)}')

train_pairs = [(inp, tgt) for inp, tgt in paired_files if get_doc_id(inp.name) in train_docs]
val_pairs   = [(inp, tgt) for inp, tgt in paired_files if get_doc_id(inp.name) in val_docs]
print(f'Train image pairs: {len(train_pairs)}, Val image pairs: {len(val_pairs)}')

In [ ]:
# ── Resize and save (always output as .jpg for uniform output dataset) ────
def resize_pair(args):
    src_inp_str, src_tgt_str, dst_inp_str, dst_tgt_str = args
    src_inp, src_tgt = Path(src_inp_str), Path(src_tgt_str)
    dst_inp, dst_tgt = Path(dst_inp_str), Path(dst_tgt_str)
    
    for src, dst in [(src_inp, dst_inp), (src_tgt, dst_tgt)]:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            continue
        with Image.open(src).convert('RGB') as img:
            img_resized = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            img_resized.save(dst, 'JPEG', quality=JPEG_Q)

tasks = []
# Training split
for inp, tgt in train_pairs:
    dst_name = inp.stem + '.jpg'
    dst_inp = OUT_DIR / 'train' / 'input' / dst_name
    dst_tgt = OUT_DIR / 'train' / 'target' / dst_name
    tasks.append((str(inp), str(tgt), str(dst_inp), str(dst_tgt)))

# Validation split
for inp, tgt in val_pairs:
    dst_name = inp.stem + '.jpg'
    dst_inp = OUT_DIR / 'val' / 'input' / dst_name
    dst_tgt = OUT_DIR / 'val' / 'target' / dst_name
    tasks.append((str(inp), str(tgt), str(dst_inp), str(dst_tgt)))

# Official test set
test_in = get_split_dir(SD7K_RAW, 'test', 'input')
test_tgt = get_split_dir(SD7K_RAW, 'test', 'target')
test_tgt_lookup = build_target_lookup(test_tgt)
test_in_files = glob_images(test_in)

test_matched = 0
for inp in test_in_files:
    tgt = find_matching_target(inp, test_tgt_lookup)
    if tgt is not None:
        dst_name = inp.stem + '.jpg'
        dst_inp = OUT_DIR / 'test' / 'input' / dst_name
        dst_tgt = OUT_DIR / 'test' / 'target' / dst_name
        tasks.append((str(inp), str(tgt), str(dst_inp), str(dst_tgt)))
        test_matched += 1

print(f'Test matched: {test_matched} pairs')
print(f'Total verified pairs to process: {len(tasks)}')

with ProcessPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(resize_pair, tasks), total=len(tasks), desc='Resizing'))

print('Done! Folder sizes:')
for split in ['train', 'val', 'test']:
    n = len(list((OUT_DIR / split / 'input').glob('*.jpg')))
    print(f'  {split}: {n} pairs')

In [ ]:
# ── Quick sanity check ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
sample = list((OUT_DIR / 'train' / 'input').glob('*.jpg'))[:3]
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for col, sp in enumerate(sample):
    s = Image.open(sp)
    c = Image.open(OUT_DIR / 'train' / 'target' / sp.name)
    axes[0, col].imshow(s); axes[0, col].set_title(sp.name[:12], fontsize=7); axes[0, col].axis('off')
    axes[1, col].imshow(c); axes[1, col].axis('off')
plt.suptitle(f'Preprocessed pairs ({IMAGE_SIZE}×{IMAGE_SIZE})')
plt.tight_layout(); plt.show()

In [ ]:
# ── (Optional) Zip output for easy download / Kaggle dataset creation ────────
if KAGGLE:
    print('Archiving /kaggle/working/sd7k_512 into zip...')
    shutil.make_archive('/kaggle/working/sd7k_512', 'zip', '/kaggle/working/sd7k_512')
    print('Saved: /kaggle/working/sd7k_512.zip')